### Workshop ➜ Homework task.

(Scroll down to the bottom for the actual questions.)

In [1]:
import dlt
from dlt.sources.rest_api import rest_api_source
from dlt.sources.rest_api.typing import PageNumberPaginatorConfig

----

#### Define the API.

In [2]:
def ny_taxi_source():
    return rest_api_source({
        "client": {
            "base_url": "https://us-central1-dlthub-analytics.cloudfunctions.net",
        },
        "resource_defaults": {
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "rides",
                "endpoint": {
                    "path": "data_engineering_zoomcamp_api",
                    "paginator": {
                        "type": "page_number",
                        "page_param": "page",
                        "base_page": 1,
                        "total_path": None,
                    },
                },
            },
        ],
    })

---

#### Define the pipeline.

In [3]:
pipeline = dlt.pipeline(
    pipeline_name="ny_taxi",
    destination="duckdb",
    dataset_name="ny_taxi_data",
    progress="log",
)

---

#### Extract.

In [4]:
extract_info = pipeline.extract(ny_taxi_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 176.41 MB (60.00%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.81s | Rate: 0.00/s
rides: 1000  | Time: 0.00s | Rate: 233016888.89/s
Memory usage: 182.30 MB (60.10%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.38s | Rate: 0.00/s
rides: 2000  | Time: 1.58s | Rate: 1267.22/s
Memory usage: 183.92 MB (60.00%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 5.20s | Rate: 0.00/s
rides: 3000  | Time: 3.40s | Rate: 882.44/s
Memory usage: 185.52 MB (59.90%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 7.17s | Ra

---

In [5]:
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()

Resources: ['rides']
Tables: ['rides']
Load ID: 1772322483.602602

Resource: rides
rows extracted: 10000



---

#### Normalization.

In [6]:
normalize_info = pipeline.normalize()
load_id = normalize_info.loads_ids[-1]
m = normalize_info.metrics[load_id][0]

print("Load ID:", load_id)
print()

print("Tables created/updated:")
for table_name, tm in m["table_metrics"].items():
    if table_name.startswith("_dlt"):
        continue
    print(f"  - {table_name}: {tm.items_count} rows")

------------------- Normalize rest_api in 1772322483.602602 --------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 195.00 MB (59.90%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772322483.602602 --------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 195.00 MB (59.90%) | CPU usage: 0.00%



2026-02-28 18:48:24,365|[WARNING]|14126|8583897664|dlt|validate.py|verify_normalized_table:91|In schema `rest_api`: The following columns in table 'rides' did not receive any data during this load and therefore could not have their types inferred:
  - rate_code
  - mta_tax

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'rate_code': {'data_type': 'text'}})



------------------- Normalize rest_api in 1772322483.602602 --------------------
Files: 1/1 (100.0%) | Time: 0.48s | Rate: 2.09/s
Items: 10000  | Time: 0.48s | Rate: 20865.90/s
Memory usage: 209.75 MB (60.00%) | CPU usage: 0.00%

Load ID: 1772322483.602602

Tables created/updated:
  - rides: 10000 rows


---

#### Load.

In [7]:
load_info = pipeline.load()

--------------------- Load rest_api in 1772321606.3875048 ----------------------
Jobs: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 222.59 MB (60.00%) | CPU usage: 0.00%

--------------------- Load rest_api in 1772321606.3875048 ----------------------
Jobs: 1/1 (100.0%) | Time: 0.05s | Rate: 18.34/s
Memory usage: 233.92 MB (60.00%) | CPU usage: 0.00%

---------------------- Load rest_api in 1772321867.246119 ----------------------
Jobs: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 233.92 MB (60.60%) | CPU usage: 0.00%

---------------------- Load rest_api in 1772321867.246119 ----------------------
Jobs: 1/1 (100.0%) | Time: 0.94s | Rate: 1.06/s
Memory usage: 275.33 MB (60.50%) | CPU usage: 0.00%

--------------------- Load rest_api in 1772322071.8613591 ----------------------
Jobs: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 275.33 MB (60.50%) | CPU usage: 0.00%

--------------------- Load rest_api in 1772322071.8613591 ----------------------
Jobs: 1/1 (100

---

#### Data inspection.

In [8]:
ds = pipeline.dataset()
ds.tables

['rides', '_dlt_version', '_dlt_loads', '_dlt_pipeline_state']

In [10]:
df = ds.rides.df()  
df.head(10)

,end_lat,end_lon,fare_amt,passenger_count,payment_type,start_lat,start_lon,tip_amt,tolls_amt,total_amt,trip_distance,trip_dropoff_date_time,trip_pickup_date_time,surcharge,vendor_name,_dlt_load_id,_dlt_id,store_and_forward
0,40.742963,-73.980072,45.0,1,Credit,40.641525,-73.787442,9.0,4.15,58.15,17.52,2009-06-14 23:48:00+00:00,2009-06-14 23:23:00+00:00,0.0,VTS,1772322483.602602,OsHoG67L68JYRQ,NaN
1,40.740187,-74.005698,6.5,1,Credit,40.722065,-74.009767,1.0,0.00,8.50,1.56,2009-06-18 17:43:00+00:00,2009-06-18 17:35:00+00:00,1.0,VTS,1772322483.602602,1wfTX8ABKKk9gg,NaN
2,40.718043,-74.004745,12.5,5,Credit,40.761945,-73.983038,2.0,0.00,15.50,3.37,2009-06-10 18:27:00+00:00,2009-06-10 18:08:00+00:00,1.0,VTS,1772322483.602602,UoaA3wrHyvGDVA,NaN
3,40.739637,-73.985233,4.9,1,CASH,40.749802,-73.992247,0.0,0.00,5.40,1.11,2009-06-14 23:58:00+00:00,2009-06-14 23:54:00+00:00,0.5,VTS,1772322483.602602,96vi1nN2i9IT6w,NaN
4,40.730032,-73.852693,25.7,1,CASH,40.776825,-73.949233,0.0,4.15,29.85,11.09,2009-06-13 13:23:00+00:00,2009-06-13 13:01:00+00:00,0.0,VTS,1772322483.602602,PdYpXI4JeTXGLw,NaN
5,40.777537,-73.976860,7.3,2,Credit,40.790582,-73.953652,2.0,0.00,10.30,2.10,2009-06-10 19:52:00+00:00,2009-06-10 19:43:00+00:00,1.0,VTS,1772322483.602602,f2HrI+TK71mQqQ,NaN
6,40.770277,-73.962125,3.7,1,Credit,40.767147,-73.966408,1.0,0.00,5.20,0.40,2009-06-10 20:09:00+00:00,2009-06-10 20:06:00+00:00,0.5,VTS,1772322483.602602,yvE6qft05bNHRw,NaN
7,40.774043,-73.951465,8.1,2,CASH,40.761750,-73.977773,0.0,0.00,8.60,2.24,2009-06-14 21:08:00+00:00,2009-06-14 20:57:00+00:00,0.5,VTS,1772322483.602602,K3ywGQdgrR0sFA,NaN
8,40.777985,-73.943683,6.1,1,CASH,40.766355,-73.959832,0.0,0.00,6.10,1.48,2009-06-14 12:56:00+00:00,2009-06-14 12:49:00+00:00,0.0,VTS,1772322483.602602,FV6AgIWEZStBOw,NaN
9,40.720052,-74.009823,8.9,1,CASH,40.751327,-73.987588,0.0,0.00,9.90,2.72,2009-06-10 18:13:00+00:00,2009-06-10 18:03:00+00:00,1.0,VTS,1772322483.602602,8dyfo0307IXBog,NaN


---

(Note: you must close the existing database connection before proceeding!)

---

### Homework Questions

In [2]:
import duckdb
conn = duckdb.connect("ny_taxi.duckdb", read_only=True)

---

#### **Q1.** What is the start date and end date of the dataset?

In [3]:
conn.execute("SELECT MIN(trip_pickup_date_time), MAX(trip_pickup_date_time) FROM ny_taxi_data.rides").fetchall()

[(datetime.datetime(2009, 6, 1, 7, 33, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>),
  datetime.datetime(2009, 6, 30, 19, 58, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>))]

Answer: `2009-06-01 to 2009-07-01`

---

#### **Q2.** What proportion of trips are paid with credit card?

In [4]:
conn.execute("""
SELECT COUNT(*) FILTER (WHERE payment_type = 'Credit') * 100.0 / COUNT(*) 
FROM ny_taxi_data.rides
""").fetchall()

[(26.66,)]

Answer: `26.66%`

---

#### **Q3.** What is the total amount of money generated in tips?

In [5]:
conn.execute("SELECT SUM(tip_amt) FROM ny_taxi_data.rides").fetchall()

[(6063.410000000009,)]

Answer: `$6,063.41`

---